In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
import pandas as pd
from scipy import interpolate
from scipy.stats import linregress
from voigt_fit_lib import *

In [2]:
plt.style.use('/Users/thepoetoftwilight/Documents/CUBS/Code/science.mplstyle')

Read CSV

In [3]:
df = pd.read_csv('/Users/thepoetoftwilight/Documents/CUBS/Data/HST_UVQSOspec_v600_2024_01.csv')

In [4]:
df_FUV_NUV = df.loc[(df['COSwG130M']!=0)&(df['COSwG160M']!=0)&((df['COSwG185M']!=0)|(df['STISwE230M']!=0)|(df['COSwG225M']!=0))]

In [5]:
df_FUV_NUV_quick = df_FUV_NUV[['z', 'Name', 'COSwG185M', 'STISwE230M', 'COSwG225M']]

In [6]:
df_FUV_NUV_quick

,z,Name,COSwG185M,STISwE230M,COSwG225M
0,0.00502,MCG+00-25-010,2168.064,0.0,0.0
1,0.00230,LEDA 2816013,404.064,0.0,0.0
3,0.03386,Mrk 1486,1776.064,0.0,0.0
6,0.02130,[BKD2008] WR 276,1828.096,0.0,0.0
7,0.00388,LEDA 29998,356.064,0.0,0.0
...,...,...,...,...,...
894,0.57023,QSO J0401-0540,7825.600,0.0,0.0
895,0.58800,QSO B0946+487,8598.592,0.0,0.0
897,0.32500,LBQS 1340-0038,24297.120,0.0,0.0
898,0.87358,PB 2049,8122.592,0.0,0.0


Classify into cases

In [7]:
def classify(z, x185, x230, x225):
    
    if x185!=0 and x230==0 and x225==0:
        return 1
    
    elif x185==0 and x230!=0 and x225==0:
        return 2
    
    elif x185==0 and x230==0 and x225!=0:
        return 3
    
    elif x185!=0 and x230!=0 and x225==0:
        return 4
    
    elif x185==0 and x230!=0 and x225!=0:
        return 5
    
    elif x185!=0 and x230==0 and x225!=0:
        return 6
    
    else:
        return 7

In [8]:
zQ_arr = np.array(df_FUV_NUV_quick['z'])
G185_arr = np.array(df_FUV_NUV_quick['COSwG185M'])
E230_arr = np.array(df_FUV_NUV_quick['STISwE230M'])
G225_arr = np.array(df_FUV_NUV_quick['COSwG225M'])
case_arr = np.zeros(len(zQ_arr), dtype=int)

In [9]:
for i in range(len(zQ_arr)):
    case_arr[i] = classify(zQ_arr[i], G185_arr[i], E230_arr[i], G225_arr[i])

Compute blind search redshift pathlength. First specify accessible redshift ranges.

In [10]:
zmax_FUV = 0.15

zmin_G185 = 0.17
zmax_G185 = 0.35

zmin_E230 = 0.29
zmax_E230 = 0.79

zmin_G225 = 0.44
zmax_G225 = 0.63

In [11]:
def dz_blind(z, x185, x230, x225, c):
    
    if c == 1:
        if z<zmin_G185:
            return min(z,zmax_FUV) # Only FUV
        else:
            return min(z,zmax_G185)-zmin_G185+zmax_FUV # Already searched FUV, start G185M
        
    elif c == 2:
        if z<zmin_E230:
            return min(z,zmax_FUV) # Only FUV
        else:
            return min(z,zmax_E230)-zmin_E230+zmax_FUV # Already searched FUV, start E230M
     
    elif c == 3:
        if z<zmin_G225:
            return min(z,zmax_FUV) # Only FUV
        else:
            return min(z,zmax_G225)-zmin_G225+zmax_FUV # Already searched FUV, start G225M   
        
    elif c==4:
        if z<zmin_G185:
            return min(z,zmax_FUV) # Only FUV
        else:
            return min(z,zmax_E230)-zmin_G185+zmax_FUV # Pretend G185M + E230M is a single grating
        
    elif c==5:
        if z<zmin_E230:
            return min(z,zmax_FUV)
        else:
            return min(z,zmax_E230)-zmin_E230+zmax_FUV # Ignore G225M
        
    elif c==6:
        if z<zmin_G185:
            return min(z,zmax_FUV)
        elif zmin_G185<z<zmin_G225:
            return min(z,zmax_G185)-zmin_G185+zmax_FUV # No consideration for G225M yet
        else:
            return min(z,zmax_G225)-zmin_G225+(zmax_G185-zmin_G185)+zmax_FUV # Already searched G185M
        
    elif c==7:
        if z<zmin_G185:
            return min(z,zmax_FUV) # Only FUV
        else:
            return min(z,zmax_E230)-zmin_G185+zmax_FUV # Pretend G185M + E230M is a single grating, ignore G225M

In [12]:
dz_blind_arr = np.zeros(len(zQ_arr))

for i in range(len(zQ_arr)):
    dz_blind_arr[i] = dz_blind(zQ_arr[i], G185_arr[i], E230_arr[i], G225_arr[i], case_arr[i])

In [13]:
dN_dz = 6.2

In [14]:
dz_blind_arr

array([0.00502   , 0.0023    , 0.03386   , 0.0213    , 0.00388   ,
       0.03787   , 0.33      , 0.00268   , 0.01485   , 0.52      ,
       0.33      , 0.0209    , 0.77      , 0.34      , 0.002232  ,
       0.04722   , 0.00253   , 0.0984    , 0.005     , 0.005405  ,
       0.0135    , 0.09116   , 0.16068   , 0.10212   , 0.02005   ,
       0.33      , 0.33      , 0.03329   , 0.1265    , 0.00446   ,
       0.004721  , 0.01764   , 0.12695   , 0.04515   , 0.0025    ,
       0.06683   , 0.33      , 0.33      , 0.0227    , 0.00288   ,
       0.02246   , 0.33      , 0.12316   , 0.16167   , 0.1536    ,
       0.33      , 0.0274    , 0.33      , 0.09426   , 0.07579   ,
       0.33      , 0.14914   , 0.33      , 0.77      , 0.003903  ,
       0.03040355, 0.035     , 0.008     , 0.009755  , 0.01627   ,
       0.18553   , 0.52      , 0.77      , 0.3113    , 0.52      ,
       0.77      , 0.77      , 0.15      , 0.15      , 0.391     ,
       0.1800237 , 0.33      , 0.40429   , 0.24428   , 0.52   

In [15]:
dN_dz*np.sum(dz_blind_arr)

99.46544215000002

Require CIII in search $z_\mathrm{CIV}>0.21$

In [16]:
zmin_CIII = 0.21

In [17]:
def dz_CIII(z, x185, x230, x225, c):
    
    if c == 1:
        if z<zmin_CIII:
            return 0 # Cannot search in FUV
        else:
            return min(z,zmax_G185)-zmin_CIII # Only begin looking at zmin_CIII
        
    elif c == 2:
        if z<zmin_E230:
            return 0 # Nothing to find
        else:
            return min(z,zmax_E230)-zmin_E230 # Cannot search FUV, start E230M
     
    elif c == 3:
        if z<zmin_G225:
            return 0
        else:
            return min(z,zmax_G225)-zmin_G225
        
    elif c==4:
        if z<zmin_CIII:
            return 0 # Only FUV
        else:
            return min(z,zmax_E230)-zmin_CIII # Pretend G185M + E230M is a single grating
        
    elif c==5:
        if z<zmin_E230:
            return 0
        else:
            return min(z,zmax_E230)-zmin_E230 # Ignore G225M
        
    elif c==6:
        if z<zmin_CIII:
            return 0
        elif zmin_CIII<z<zmin_G225:
            return min(z,zmax_G185)-zmin_CIII # No consideration for G225M yet
        else:
            return min(z,zmax_G225)-zmin_G225+(zmax_G185-zmin_CIII) # Already searched G185M
        
    elif c==7:
        if z<zmin_CIII:
            return 0
        else:
            return min(z,zmax_E230)-zmin_CIII # Pretend G185M + E230M is a single grating, ignore G225M

In [18]:
dz_CIII_arr = np.zeros(len(zQ_arr))

for i in range(len(zQ_arr)):
    dz_CIII_arr[i] = dz_CIII(zQ_arr[i], G185_arr[i], E230_arr[i], G225_arr[i], case_arr[i])

In [19]:
dz_CIII_arr

array([0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.14   ,
       0.     , 0.     , 0.33   , 0.14   , 0.     , 0.58   , 0.19   ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.14   , 0.14   , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.14   , 0.14   , 0.     , 0.     , 0.     , 0.14   ,
       0.     , 0.     , 0.     , 0.14   , 0.     , 0.14   , 0.     ,
       0.     , 0.14   , 0.     , 0.14   , 0.58   , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.33   , 0.58   ,
       0.1213 , 0.33   , 0.58   , 0.58   , 0.     , 0.     , 0.201  ,
       0.     , 0.14   , 0.21429, 0.05428, 0.33   , 0.14   , 0.14   ,
       0.14   , 0.115  , 0.14   , 0.     ])

In [20]:
dN_dz*np.sum(dz_CIII_arr)

45.606394

Require OIV ($z_\mathrm{CIV}>0.5$)

In [21]:
zmin_OIV = 0.5

In [22]:
def dz_OIV(z, x185, x230, x225, c):
    
    if c == 1:
        return 0 # Cannot search in FUV or G185M
        
    elif c == 2:
        if z<zmin_OIV:
            return 0 # Nothing to find
        else:
            return min(z,zmax_E230)-zmin_OIV # Begin at zmin_OIV in E230M
     
    elif c == 3:
        if z<zmin_OIV:
            return 0
        else:
            return min(z,zmax_G225)-zmin_OIV # Begin at zmin_OIV in G225M
        
    elif c==4:
        if z<zmin_OIV:
            return 0
        else:
            return min(z,zmax_E230)-zmin_OIV # Ignore G185M
        
    elif c==5:
        if z<zmin_OIV:
            return 0
        else:
            return min(z,zmax_E230)-zmin_OIV # Ignore G225M
        
    elif c==6:
        if z<zmin_OIV:
            return 0
        else:
            return min(z,zmax_G225)-zmin_OIV # Ignore G185M
        
    elif c==7:
        if z<zmin_OIV:
            return 0
        else:
            return min(z,zmax_E230)-zmin_OIV # Ignore G185M, G225M

In [23]:
dz_OIV_arr = np.zeros(len(zQ_arr))

for i in range(len(zQ_arr)):
    dz_OIV_arr[i] = dz_OIV(zQ_arr[i], G185_arr[i], E230_arr[i], G225_arr[i], case_arr[i])

In [24]:
dz_OIV_arr

array([0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.13   , 0.     , 0.     , 0.29   , 0.13   ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.29   , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.13   , 0.29   ,
       0.     , 0.13   , 0.29   , 0.29   , 0.     , 0.     , 0.001  ,
       0.     , 0.     , 0.01429, 0.     , 0.13   , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     ])

In [25]:
dN_dz*np.sum(dz_OIV_arr)

13.114798000000002